In [ ]:
import pandas as pd

train = pd.read_parquet("../artifacts/ml_table_train.parquet")
val = pd.read_parquet("../artifacts/ml_table_val.parquet")
test = pd.read_parquet("../artifacts/ml_table_test.parquet")

In [ ]:
print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

In [ ]:
X_train = train.drop(columns=["is_late"])
y_train = train["is_late"]

X_val = val.drop(columns=["is_late"]) 
y_val = val["is_late"]

X_test = test.drop(columns=["is_late"])
y_test = test["is_late"]

In [ ]:
X_train.columns.tolist()

In [ ]:
id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

potential_leakage_columns = [
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "avg_review_score",
    "review_count",
    "combined_review_comments",
    "latest_review_date"
]

In [ ]:
X_train.select_dtypes(include=['datetime64']).columns

In [ ]:
X_train['order_status'].unique()

In [ ]:
X_train["order_status"].value_counts()

### order_status


- `delivered`: 67,224
- `shipped`: 855
- `unavailable`: 561
- `canceled`: 429
- `processing`: 290
- `invoiced`: 242
- `created`: 5
- `approved`: 2

`order_status` was excluded because it represents the order's later status rather than the information available at prediction time.

Most orders are recorded as `delivered`, while smaller numbers are `shipped`, `canceled`, `processing`, and other statuses. Since these statuses occur during or after the order lifecycle, using them to predict delivery lateness would introduce data leakage.

## Feature Selection Based on Prediction Time

### Keep

- `order_purchase_timestamp`
- `order_approved_at`
- `order_estimated_delivery_date`
- `shipping_limit_date`

### Drop

- `order_delivered_carrier_date`
- `order_delivered_customer_date`
- `latest_review_date`
- `avg_review_score`
- `review_count`
- `combined_review_comments`
- `order_status`

### ID Columns — Drop

- `order_id`
- `customer_id`
- `customer_unique_id`

In [ ]:
columns_to_drop = [
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "latest_review_date",
    "avg_review_score",
    "review_count",
    "combined_review_comments",
    "order_id",
    "customer_id",
    "customer_unique_id"
]

In [ ]:
X_train = X_train.drop(columns=columns_to_drop)
X_val = X_val.drop(columns=columns_to_drop)
X_test = X_test.drop(columns=columns_to_drop)



In [ ]:
print("X_train shape:", X_train.shape)
print("Remaining columns:", X_train.columns.tolist())

## preprocessing

In [ ]:
from sklearn.impute import SimpleImputer



- date_features

In [121]:
date_features = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date",
    "shipping_limit_date"
]

def extract_date_features(df):
    df_dates = pd.DataFrame(index=df.index)

    if "order_purchase_timestamp" in df.columns and "order_estimated_delivery_date" in df.columns:
        df_dates["purchase_to_estimated_days"] = (
            df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
        ).dt.total_seconds() / 86400.0



    if "order_purchase_timestamp" in df.columns:
        df_dates["purchase_year"] = df["order_purchase_timestamp"].dt.year
        df_dates["purchase_month"] = df["order_purchase_timestamp"].dt.month
        df_dates["purchase_day"] = df["order_purchase_timestamp"].dt.day
        df_dates["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek  # 0=Monday, 6=Sunday
        df_dates["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
        
    return df_dates

# تطبيق الدالة على المجموعات الثلاثة (Train, Validation, Test)
X_train_dates_final = extract_date_features(X_train)
X_val_dates_final = extract_date_features(X_val)
X_test_dates_final = extract_date_features(X_test)

print("✅ Date features extracted successfully!")
print("Train dates shape:", X_train_dates_final.shape)
print("Columns created:", X_train_dates_final.columns.tolist())


✅ Date features extracted successfully!
Train dates shape: (69608, 6)
Columns created: ['purchase_to_estimated_days', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_dayofweek', 'purchase_hour']


- numeric_features

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()

print(numeric_features)
print("Number of numerical features:", len(numeric_features))

In [ ]:
numeric_missing = X_train[numeric_features].isna().sum()

print(numeric_missing[numeric_missing > 0])

In [ ]:
missing_count = X_train["payment_value"].isna().sum()
missing_percentage = X_train["payment_value"].isna().mean() * 100

print("Missing values:", missing_count)
print("Missing percentage:", missing_percentage)

In [ ]:
payment_value_median = X_train["payment_value"].median()

print("Payment value median:", payment_value_median)

In [ ]:
geographic_features = [
    "customer_lat",
    "customer_lng",
    "seller_lat",
    "seller_lng",
    "distance_km"
]
missing_indicator_features = [
    "shipping_limit_missing"
]
numeric_non_geo_features = [
    col for col in numeric_features
    if col not in geographic_features
    if col not in missing_indicator_features
]

print(numeric_non_geo_features)
print("Number of non-geographic numerical features:", len(numeric_non_geo_features))

In [ ]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(strategy="median")

numeric_imputer.fit(X_train[numeric_non_geo_features])

In [ ]:
X_train_numeric_imputed = numeric_imputer.transform(
    X_train[numeric_non_geo_features]
)

In [ ]:
print("Shape:", X_train_numeric_imputed.shape)
print("Missing values:", pd.isna(X_train_numeric_imputed).sum())

In [ ]:
X_train_numeric_imputed = pd.DataFrame(
    X_train_numeric_imputed,
    columns=numeric_non_geo_features,
    index=X_train.index
)
X_train_numeric_imputed.head()

In [ ]:
X_val_numeric_imputed = numeric_imputer.transform(
    X_val[numeric_non_geo_features]
)

X_test_numeric_imputed = numeric_imputer.transform(
    X_test[numeric_non_geo_features]
)
X_val_numeric_imputed = pd.DataFrame(
    X_val_numeric_imputed,
    columns=numeric_non_geo_features,
    index=X_val.index
)

X_test_numeric_imputed = pd.DataFrame(
    X_test_numeric_imputed,
    columns=numeric_non_geo_features,
    index=X_test.index
)
print("Train:", X_train_numeric_imputed.shape)
print("Validation:", X_val_numeric_imputed.shape)
print("Test:", X_test_numeric_imputed.shape)

print("Train missing:", X_train_numeric_imputed.isna().sum().sum())
print("Validation missing:", X_val_numeric_imputed.isna().sum().sum())
print("Test missing:", X_test_numeric_imputed.isna().sum().sum())

In [ ]:
from sklearn.preprocessing import RobustScaler
import pandas as pd


robust_scaler = RobustScaler()
robust_scaler.fit(X_train_numeric_imputed)


X_train_scaled_array = robust_scaler.transform(X_train_numeric_imputed)
X_val_scaled_array = robust_scaler.transform(X_val_numeric_imputed)
X_test_scaled_array = robust_scaler.transform(X_test_numeric_imputed)

X_train_numeric_final = pd.DataFrame(
    X_train_scaled_array, 
    columns=numeric_non_geo_features, 
    index=X_train.index
)

X_val_numeric_final = pd.DataFrame(
    X_val_scaled_array, 
    columns=numeric_non_geo_features, 
    index=X_val.index
)

X_test_numeric_final = pd.DataFrame(
    X_test_scaled_array, 
    columns=numeric_non_geo_features, 
    index=X_test.index
)


print(" RobustScaler applied and saved successfully!")
print("Train scaled shape:", X_train_numeric_final.shape)
print("Val scaled shape:", X_val_numeric_final.shape)
print("Test scaled shape:", X_test_numeric_final.shape)

 RobustScaler applied and saved successfully!
Train scaled shape: (69608, 12)
Val scaled shape: (14916, 12)
Test scaled shape: (14917, 12)


In [117]:
X_train_numeric_final.describe().T[["min", "max"]]

,min,max
payment_value,-0.825931,118.788699
payment_count,0.000000,28.000000
max_payment_installments,-0.333333,7.333333
total_items_count,0.000000,20.000000
total_items_price,-0.800096,128.408654
total_freight_value,-1.758115,103.193717
total_product_weight_g,-0.444444,102.000000
product_length_cm,-0.900000,4.000000
product_height_cm,-0.916667,7.666667
product_width_cm,-0.933333,6.533333


- categorical_features

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(categorical_features)
print("Number of categorical features:", len(categorical_features))

customer_zip_code_prefix      13872                  drop
customer_city                  3780                  Grouping rare categories + One-Hot
sellers_zip_codes_combined     2331                   drop 
primary_seller_zip             1754                   drop
sellers_cities_combined         954                
sellers_states_combined          75
primary_product_category         73                  
customer_state                   27                   
payment_type                      6    عدد الاعمدة قليل   

In [106]:
city_counts = X_train["customer_city"].value_counts()

common_cities = city_counts[city_counts >= 10].index
X_train["customer_city_grouped"] = X_train["customer_city"].where(
    X_train["customer_city"].isin(common_cities),
    "Other"
)
X_train["customer_city_grouped"] = X_train["customer_city"].where(
    X_train["customer_city"].isin(common_cities),
    "Other"
)
X_val["customer_city_grouped"] = X_val["customer_city"].where(
    X_val["customer_city"].isin(common_cities),
    "Other"
)

X_test["customer_city_grouped"] = X_test["customer_city"].where(
    X_test["customer_city"].isin(common_cities),
    "Other"
)
from sklearn.preprocessing import OneHotEncoder

city_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
city_encoder.fit(
    X_train[["customer_city_grouped"]]
)

X_train_city_encoded = city_encoder.transform(
    X_train[["customer_city_grouped"]]
)

X_val_city_encoded = city_encoder.transform(
    X_val[["customer_city_grouped"]]
)

X_test_city_encoded = city_encoder.transform(
    X_test[["customer_city_grouped"]]
)

city_feature_names = city_encoder.get_feature_names_out(
    ["customer_city_grouped"]
)

print("Number of feature names:", len(city_feature_names))
print(city_feature_names[:10])


X_train_city_encoded = pd.DataFrame(
    X_train_city_encoded,
    columns=city_feature_names,
    index=X_train.index
)

X_val_city_encoded = pd.DataFrame(
    X_val_city_encoded,
    columns=city_feature_names,
    index=X_val.index
)

X_test_city_encoded = pd.DataFrame(
    X_test_city_encoded,
    columns=city_feature_names,
    index=X_test.index
)

Number of feature names: 767
['customer_city_grouped_Other' 'customer_city_grouped_adamantina'
 'customer_city_grouped_afonso claudio' 'customer_city_grouped_agua boa'
 'customer_city_grouped_aguas de lindoia'
 'customer_city_grouped_aguas lindas de goias'
 'customer_city_grouped_agudos' 'customer_city_grouped_alagoinhas'
 'customer_city_grouped_alegre' 'customer_city_grouped_alegrete']


In [112]:
temp = X_train[["payment_type"]].copy()
temp["is_late"] = y_train

temp.groupby("payment_type")["is_late"].mean()

payment_type
boleto                  0.090928
credit_card             0.087017
credit_card, voucher    0.062954
debit_card              0.096234
voucher                 0.073235
voucher, credit_card    0.085781
Name: is_late, dtype: float64

In [ ]:

payment_encoder = OneHotEncoder(
    handle_unknown="ignore", 
    sparse_output=False, 
    drop="first"
)

payment_encoder.fit(X_train[["payment_type"]])

X_train_payment_encoded = payment_encoder.transform(X_train[["payment_type"]])
X_val_payment_encoded = payment_encoder.transform(X_val[["payment_type"]])
X_test_payment_encoded = payment_encoder.transform(X_test[["payment_type"]])

payment_feature_names = payment_encoder.get_feature_names_out(["payment_type"])

X_train_payment_final = pd.DataFrame(
    X_train_payment_encoded, 
    columns=payment_feature_names, 
    index=X_train.index
)

X_val_payment_final = pd.DataFrame(
    X_val_payment_encoded, 
    columns=payment_feature_names, 
    index=X_val.index
)

X_test_payment_final = pd.DataFrame(
    X_test_payment_encoded, 
    columns=payment_feature_names, 
    index=X_test.index
)

print(" payment_type encoded successfully !")
print("Created columns:", payment_feature_names.tolist())
print("Train encoded shape:", X_train_payment_final.shape)

 payment_type encoded successfully with Logistic Regression optimization!
Created columns: ['payment_type_credit_card', 'payment_type_credit_card, voucher', 'payment_type_debit_card', 'payment_type_voucher', 'payment_type_voucher, credit_card', 'payment_type_nan']
Train encoded shape: (69608, 6)


c:\Users\shima\OneDrive\Desktop\task 2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\shima\OneDrive\Desktop\task 2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [127]:
# 1. إنشاء الـ Encoder لـ customer_state مع استخدام drop='first'
state_encoder = OneHotEncoder(
    handle_unknown="ignore", 
    sparse_output=False, 
    drop="first"
)

# 2. عمل Fit على مجموعة التدريب (Train) فقط لمنع تسريب البيانات
state_encoder.fit(X_train[["customer_state"]])

# 3. تطبيق التحويل على الجداول الثلاثة
X_train_state_encoded = state_encoder.transform(X_train[["customer_state"]])
X_val_state_encoded = state_encoder.transform(X_val[["customer_state"]])
X_test_state_encoded = state_encoder.transform(X_test[["customer_state"]])

# 4. استخراج أسماء الأعمدة الجديدة
state_feature_names = state_encoder.get_feature_names_out(["customer_state"])

# 5. تحويل النتائج إلى DataFrames مرتبة بنفس الـ Index
X_train_state_final = pd.DataFrame(
    X_train_state_encoded, 
    columns=state_feature_names, 
    index=X_train.index
)

X_val_state_final = pd.DataFrame(
    X_val_state_encoded, 
    columns=state_feature_names, 
    index=X_val.index
)

X_test_state_final = pd.DataFrame(
    X_test_state_encoded, 
    columns=state_feature_names, 
    index=X_test.index
)

print("customer_state encoded successfully!")
print("Train encoded shape:", X_train_state_final.shape)

customer_state encoded successfully!
Train encoded shape: (69608, 26)


In [128]:
# 1. إنشاء الـ Encoder لـ primary_product_category
category_encoder = OneHotEncoder(
    handle_unknown="ignore", 
    sparse_output=False, 
    drop="first"
)

# 2. عمل Fit على مجموعة التدريب (Train) فقط لمنع تسريب البيانات
category_encoder.fit(X_train[["primary_product_category"]])

# 3. تطبيق التحويل على الجداول الثلاثة
X_train_category_encoded = category_encoder.transform(X_train[["primary_product_category"]])
X_val_category_encoded = category_encoder.transform(X_val[["primary_product_category"]])
X_test_category_encoded = category_encoder.transform(X_test[["primary_product_category"]])

# 4. استخراج أسماء الأعمدة الجديدة
category_feature_names = category_encoder.get_feature_names_out(["primary_product_category"])

# 5. تحويل النتائج إلى DataFrames مرتبة بنفس الـ Index
X_train_category_final = pd.DataFrame(
    X_train_category_encoded, 
    columns=category_feature_names, 
    index=X_train.index
)

X_val_category_final = pd.DataFrame(
    X_val_category_encoded, 
    columns=category_feature_names, 
    index=X_val.index
)

X_test_category_final = pd.DataFrame(
    X_test_category_encoded, 
    columns=category_feature_names, 
    index=X_test.index
)

print(" primary_product_category encoded successfully!")
print("Train encoded shape:", X_train_category_final.shape)

 primary_product_category encoded successfully!
Train encoded shape: (69608, 73)


c:\Users\shima\OneDrive\Desktop\task 2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\shima\OneDrive\Desktop\task 2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [ ]:
# 1. إنشاء الـ Encoder لـ sellers_states_combined
seller_state_encoder = OneHotEncoder(
    handle_unknown="ignore", 
    sparse_output=False, 
    drop="first"
)

# 2. التدريب على مجموعة التدريب فقط (Train)
seller_state_encoder.fit(X_train[["sellers_states_combined"]])

# 3. تحويل الجداول الثلاثة
X_train_s_state_encoded = seller_state_encoder.transform(X_train[["sellers_states_combined"]])
X_val_s_state_encoded = seller_state_encoder.transform(X_val[["sellers_states_combined"]])
X_test_s_state_encoded = seller_state_encoder.transform(X_test[["sellers_states_combined"]])

# 4. استخراج أسماء الأعمدة الجديدة
seller_state_feature_names = seller_state_encoder.get_feature_names_out(["sellers_states_combined"])

# 5. تحويل النتائج إلى DataFrames مرتبة بنفس الـ Index
X_train_seller_state_final = pd.DataFrame(
    X_train_s_state_encoded, 
    columns=seller_state_feature_names, 
    index=X_train.index
)

X_val_seller_state_final = pd.DataFrame(
    X_val_s_state_encoded, 
    columns=seller_state_feature_names, 
    index=X_val.index
)

X_test_seller_state_final = pd.DataFrame(
    X_test_s_state_encoded, 
    columns=seller_state_feature_names, 
    index=X_test.index
)

print(" sellers_states_combined encoded successfully!")
print("Train encoded shape:", X_train_seller_state_final.shape)

✅ sellers_states_combined encoded successfully!
Train encoded shape: (69608, 75)


c:\Users\shima\OneDrive\Desktop\task 2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\shima\OneDrive\Desktop\task 2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [130]:
# 1. حساب تكرار مدن البائعين في مجموعة التدريب
seller_city_counts = X_train["sellers_cities_combined"].value_counts()
threshold = 10
common_seller_cities = seller_city_counts[seller_city_counts >= threshold].index

# 2. تجميع الفئات النادرة إلى "Other" في الجداول الثلاثة
X_train["seller_city_grouped"] = X_train["sellers_cities_combined"].where(
    X_train["sellers_cities_combined"].isin(common_seller_cities), "Other"
)
X_val["seller_city_grouped"] = X_val["sellers_cities_combined"].where(
    X_val["sellers_cities_combined"].isin(common_seller_cities), "Other"
)
X_test["seller_city_grouped"] = X_test["sellers_cities_combined"].where(
    X_test["sellers_cities_combined"].isin(common_seller_cities), "Other"
)

# 3. إنشاء الـ OneHotEncoder مع drop='first'
seller_city_encoder = OneHotEncoder(
    handle_unknown="ignore", 
    sparse_output=False, 
    drop="first"
)
seller_city_encoder.fit(X_train[["seller_city_grouped"]])

# 4. تحويل الجداول الثلاثة
X_train_s_city_encoded = seller_city_encoder.transform(X_train[["seller_city_grouped"]])
X_val_s_city_encoded = seller_city_encoder.transform(X_val[["seller_city_grouped"]])
X_test_s_city_encoded = seller_city_encoder.transform(X_test[["seller_city_grouped"]])

# 5. استخراج الأسماء وتحويلها إلى DataFrames
seller_city_feature_names = seller_city_encoder.get_feature_names_out(["seller_city_grouped"])

X_train_seller_city_final = pd.DataFrame(
    X_train_s_city_encoded, columns=seller_city_feature_names, index=X_train.index
)
X_val_seller_city_final = pd.DataFrame(
    X_val_s_city_encoded, columns=seller_city_feature_names, index=X_val.index
)
X_test_seller_city_final = pd.DataFrame(
    X_test_s_city_encoded, columns=seller_city_feature_names, index=X_test.index
)

print("✅ sellers_cities_combined grouped and encoded successfully!")
print("Train encoded shape:", X_train_seller_city_final.shape)

✅ sellers_cities_combined grouped and encoded successfully!
Train encoded shape: (69608, 296)
